In [1]:
import os
import psycopg
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Get the connection string from the environment variable
conn_string = os.getenv("NEON_DB_STRING")

### Read data from a existing table

In [2]:
read_conn = psycopg.connect(conn_string)
print(read_conn.info)
cursor = read_conn.cursor()
print(cursor)
cursor.execute("SELECT * FROM pg_catalog.pg_tables WHERE schemaname='public';")
# cursor.execute("SELECT * FROM 'public'.'Users';")
rows = cursor.fetchall()
for row in rows:
    print(row)

read_conn.close()

<psycopg.Cursor [no result] [IDLE] (host=ep-super-art-ald8qrox-pooler.c-3.eu-central-1.aws.neon.tech user=neondb_owner database=neondb) at 0x16b4641c7d0>
('public', 'Users', 'neondb_owner', None, True, False, False, False)


In [3]:
read_conn = psycopg.connect(conn_string)
cursor = read_conn.cursor()
cursor.execute('SELECT * FROM "Users";')
rows = cursor.fetchall()
for row in rows:
    print(row)

read_conn.close()

(1, 'vikram', 'Vikram', 'Bhatt', 'vikram.bhatt@student.ie.edu', '12345678')


### Use Polars to read the data

Use this as the guide: https://docs.pola.rs/user-guide/io/database/

In [5]:
import polars as pl

query = 'SELECT * FROM "Users";'

test = pl.read_database_uri(query=query, uri=conn_string)
test.head()

id,username,firstname,lastname,email,password
i32,str,str,str,str,str
1,"""vikram""","""Vikram""","""Bhatt""","""vikram.bhatt@student.ie.edu""","""12345678"""


### Use Polars to write the data

In [12]:
df = pl.DataFrame({
        "username": "rallulu",
        "firstname": "Raluca",
        "lastname": "Gogosoiu",
        "email": "ralucagogosoiu@student.ie.edu",
        "password": "123456789",
    }
)

df.write_database(
    table_name="Users", 
    connection=conn_string, 
    engine="adbc",
    if_table_exists="append"
)

1

### Create Table in DB and add data

In [ ]:
try:
    with psycopg.connect(conn_string) as conn:
        print("Connection established")

        # Open a cursor to perform database operations
        with conn.cursor() as cur:
            # Drop the table if it already exists
            cur.execute("DROP TABLE IF EXISTS books;")
            print("Finished dropping table (if it existed).")

            # Create a new table
            cur.execute("""
                CREATE TABLE books (
                    id SERIAL PRIMARY KEY,
                    title VARCHAR(255) NOT NULL,
                    author VARCHAR(255),
                    publication_year INT,
                    in_stock BOOLEAN DEFAULT TRUE
                );
            """)
            print("Finished creating table.")

            # Insert a single book record
            cur.execute(
                "INSERT INTO books (title, author, publication_year, in_stock) VALUES (%s, %s, %s, %s);",
                ("The Catcher in the Rye", "J.D. Salinger", 1951, True),
            )
            print("Inserted a single book.")

            # Data to be inserted
            books_to_insert = [
                ("The Hobbit", "J.R.R. Tolkien", 1937, True),
                ("1984", "George Orwell", 1949, True),
                ("Dune", "Frank Herbert", 1965, False),
            ]

            # Insert multiple books at once
            cur.executemany(
                "INSERT INTO books (title, author, publication_year, in_stock) VALUES (%s, %s, %s, %s);",
                books_to_insert,
            )

            print("Inserted 3 rows of data.")
            # The transaction is committed automatically when the 'with' block exits in psycopg (v3)

except Exception as e:
    print("Connection failed.")
    print(e)